In [ ]:
import sys 
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch


In [ ]:
PIPELINE_ROOT = Path(
    "/home/rittick-dutta/Desktop/Research/Pipeline2"
)

if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

DATASET_ROOT = Path(
    "/home/rittick-dutta/Desktop/Research/"
    "symile-mimic/symile-mimic-aws"
)

In [ ]:
from utils.data_helper import TARGETS, SymileMIMICDataset, create_dataloaders

In [ ]:
print("Pipeline exists:", PIPELINE_ROOT.is_dir())
print("Dataset exists:", DATASET_ROOT.is_dir())

In [ ]:
train_loader, val_loader, test_loaders = create_dataloaders(
    dataset_root=DATASET_ROOT,
    batch_size=8,
    num_workers=0,
    seed=42,
)
test_loader = test_loaders["all"]
print("Training samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
for condition, loader in test_loaders.items():
    print(condition, len(loader.dataset), loader.dataset.modalities)

In [ ]:
batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

assert batch["labels"].shape == (batch["cxr"].size(0), len(TARGETS))
assert batch["label_mask"].shape == batch["labels"].shape
print("Prediction targets:", len(TARGETS))
print(TARGETS)

In [ ]:
# Original positive / negative / uncertain / unreported labels in each split.
for split, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    print(split)
    display(loader.dataset.label_counts())

## Five admissions from each split
Train, validation and the all-input test view contain every modality. The three test views below retain the same admissions and targets but return different input keys.

## Training


In [ ]:
train_dataset = train_loader.dataset
train_indices = list(range(min(5, len(train_dataset))))

# Retrieve once; reuse these samples in both plots below.
train_samples = [train_dataset[i] for i in train_indices]
train_table = train_dataset.metadata.iloc[train_indices].copy()
train_table["cxr_available"] = ["cxr" in s for s in train_samples]
assert train_table["hadm_id"].tolist() == [s["hadm_id"].item() for s in train_samples]
print("Training admissions:", len(train_dataset))
with pd.option_context("display.max_columns", None):
    display(train_table)


In [ ]:
# Undo ImageNet normalization for display only.
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
fig, axes = plt.subplots(1, len(train_samples), figsize=(20, 4), squeeze=False)
for ax, sample in zip(axes.flat, train_samples):
    image = sample["cxr"].numpy().transpose(1, 2, 0)
    ax.imshow(np.clip(image * std + mean, 0, 1))
    ax.set_title(f"Admission {sample['hadm_id'].item()}")
    ax.axis("off")
fig.suptitle("Train: chest X-rays — all available")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Display all 12 ECG leads for each of these same five admissions.
for sample in train_samples:
    ecg = sample["ecg"][0].numpy()  # (5000, 12)
    fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
    for lead, ax in enumerate(axes.flat):
        ax.plot(ecg[:, lead], linewidth=0.6)
        ax.set_title(f"Lead index {lead}")
        ax.set_ylabel("Scaled amplitude")
        ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("Sample index")
    fig.suptitle(f"Training: ECG — admission {sample['hadm_id'].item()}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Validation


In [ ]:
val_dataset = val_loader.dataset
val_indices = list(range(min(5, len(val_dataset))))

# Retrieve once; reuse these samples in both plots below.
val_samples = [val_dataset[i] for i in val_indices]
val_table = val_dataset.metadata.iloc[val_indices].copy()
val_table["cxr_available"] = ["cxr" in s for s in val_samples]
assert val_table["hadm_id"].tolist() == [s["hadm_id"].item() for s in val_samples]
print("Validation admissions:", len(val_dataset))
with pd.option_context("display.max_columns", None):
    display(val_table)


In [ ]:
# Undo ImageNet normalization for display only.
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
fig, axes = plt.subplots(1, len(val_samples), figsize=(20, 4), squeeze=False)
for ax, sample in zip(axes.flat, val_samples):
    image = sample["cxr"].numpy().transpose(1, 2, 0)
    ax.imshow(np.clip(image * std + mean, 0, 1))
    ax.set_title(f"Admission {sample['hadm_id'].item()}")
    ax.axis("off")
fig.suptitle("Val: chest X-rays — all available")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Display all 12 ECG leads for each of these same five admissions.
for sample in val_samples:
    ecg = sample["ecg"][0].numpy()  # (5000, 12)
    fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
    for lead, ax in enumerate(axes.flat):
        ax.plot(ecg[:, lead], linewidth=0.6)
        ax.set_title(f"Lead index {lead}")
        ax.set_ylabel("Scaled amplitude")
        ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("Sample index")
    fig.suptitle(f"Validation: ECG — admission {sample['hadm_id'].item()}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Test


In [ ]:
test_dataset = test_loader.dataset
test_indices = list(range(min(5, len(test_dataset))))

# Retrieve once; reuse these samples in both plots below.
test_samples = [test_dataset[i] for i in test_indices]
test_table = test_dataset.metadata.iloc[test_indices].copy()
test_table["cxr_available"] = ["cxr" in s for s in test_samples]
assert test_table["hadm_id"].tolist() == [s["hadm_id"].item() for s in test_samples]
print("Test admissions:", len(test_dataset))
with pd.option_context("display.max_columns", None):
    display(test_table)


In [ ]:
# Undo ImageNet normalization for display only.
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
fig, axes = plt.subplots(1, len(test_samples), figsize=(20, 4), squeeze=False)
for ax, sample in zip(axes.flat, test_samples):
    image = sample["cxr"].numpy().transpose(1, 2, 0)
    ax.imshow(np.clip(image * std + mean, 0, 1))
    ax.set_title(f"Admission {sample['hadm_id'].item()}")
    ax.axis("off")
fig.suptitle("Test: chest X-rays — all available")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Display all 12 ECG leads for each of these same five admissions.
for sample in test_samples:
    ecg = sample["ecg"][0].numpy()  # (5000, 12)
    fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
    for lead, ax in enumerate(axes.flat):
        ax.plot(ecg[:, lead], linewidth=0.6)
        ax.set_title(f"Lead index {lead}")
        ax.set_ylabel("Scaled amplitude")
        ax.grid(alpha=0.3)
    for ax in axes[-1]:
        ax.set_xlabel("Sample index")
    fig.suptitle(f"Test: ECG — admission {sample['hadm_id'].item()}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## Three test views without padding
An excluded modality is absent from the sample and its file is not opened by that view.

In [ ]:
reference = test_loaders["all"].dataset
for condition, loader in test_loaders.items():
    dataset = loader.dataset
    sample = dataset[0]
    assert np.array_equal(dataset.hadm_ids, reference.hadm_ids)
    assert np.array_equal(dataset.labels, reference.labels)
    assert np.array_equal(dataset.label_masks, reference.label_masks)
    print(condition, "input keys:", [key for key in ("cxr", "ecg", "labs") if key in sample])
    print("Admission:", sample["hadm_id"].item(), "Available arrays:", dataset.modalities)
assert "cxr" not in test_loaders["ecg_labs"].dataset[0]
assert "ecg" not in test_loaders["cxr_only"].dataset[0]
assert "labs" not in test_loaders["cxr_only"].dataset[0]